# Respuesta sugerida  Unidad-02

## 0. Carga de Librerías y Datos
Primero, importamos la librería pandas y cargamos los 3 dataset que tenemos en la misma carpeta.

In [ ]:
import pandas as pd

df_ventas = pd.read_csv('ventas.csv')
df_pedidos = pd.read_csv('pedidos.csv')
df_productos= pd.read_csv('productos.csv')

## 1. Combinamos los tres DataFrames

In [90]:
df_consolidado = pd.merge(df_ventas, df_pedidos, on='id_venta', how='left')
df_consolidado = pd.merge(df_consolidado, df_productos, on='id_producto', how='left')

## 2. Filtramos solo ventas completadas
- Usamos .copy() porque en el Paso 4 agregaremos una columna nueva 
- Esto evita el SettingWithCopyWarning y garantiza que trabajamos con un DataFrame independiente

In [91]:
df_completadas = df_consolidado[df_consolidado["estado"] == "Completado"].copy()

## 3. Identificamos los productos fantasma (ventas sin producto en catálogo)

In [92]:
productos_fantasma=  df_consolidado[ df_consolidado["nombre_producto"].isna() ]
print(f"Productos fantasma detectados: {len(productos_fantasma)}, sus id son: {productos_fantasma['id_producto'].unique()}")

Productos fantasma detectados: 1, sus id son: [5]


## 4. Creamos una columna calculada para calcular ganancia real

In [93]:
df_completadas['ganancia_real'] = df_completadas['monto_total'] * (df_completadas['margen_porcentaje'] / 100)

# imprimimos una lista de campos de nuestro df_completadas
print(list(df_completadas.columns))

['id_venta', 'id_producto', 'vendedor', 'cantidad', 'monto_total', 'estado', 'nombre_producto', 'categoria', 'margen_porcentaje', 'ganancia_real']


## 5. Consultamos el dataframe con las columnas solicitadas, con los montos redondeados y ordenado por vendedor.

In [ ]:
df_completadas[['id_venta', 'vendedor', 'nombre_producto', 'categoria', 'monto_total','cantidad', 'margen_porcentaje', 'ganancia_real']].round(2).sort_values(by='vendedor', ascending=False)

,id_venta,vendedor,nombre_producto,categoria,monto_total,cantidad,margen_porcentaje,ganancia_real
3,104,María Torres,Laptop Dell XPS,Computadoras,1200,1,18.0,216.0
6,107,María Torres,NaN,NaN,3200,1,NaN,NaN
1,102,Carlos Ruiz,Mouse Logitech,Periféricos,850,1,35.0,297.5
4,105,Carlos Ruiz,"Monitor LG 27""",Monitores,1250,5,22.0,275.0
7,108,Carlos Ruiz,Teclado Mecánico,Periféricos,300,2,28.0,84.0
0,101,Ana López,Laptop Dell XPS,Computadoras,2400,2,18.0,432.0


## 6. Agrupamos por vendedor y sumamos por monto_total, cantidad y ganancias

In [ ]:
ganancia_vendedor = df_completadas.groupby('vendedor').agg({ 'monto_total': 'sum', 'cantidad': 'sum','ganancia_real': 'sum',}).sort_values(by='ganancia_real', ascending=False)
ganancia_vendedor

,monto_total,cantidad,ganancia_real
vendedor,,,
Carlos Ruiz,2400,8,656.5
Ana López,2400,2,432.0
María Torres,4400,2,216.0


## 7. Agrupamos por categoría y calculamos el promedio de ganancia_real y monto_total

In [94]:
ganancia_categoria = df_completadas.groupby('categoria').agg({'ganancia_real': 'mean', 'monto_total': 'mean'}).sort_values(by='ganancia_real', ascending=False)
ganancia_categoria

,ganancia_real,monto_total
categoria,,
Computadoras,324.00,1800.0
Monitores,275.00,1250.0
Periféricos,190.75,575.0
